In [0]:
%pip install databricks-feature-engineering
dbutils.library.restartPython()

In [0]:
from databricks.feature_engineering import FeatureLookup
from databricks.feature_engineering import FeatureEngineeringClient

In [0]:
df=spark.read.table("dev.feature_store.sessions_arkime").select("id")
df.display()

In [0]:
fe = FeatureEngineeringClient()
def load_data(data,table_name, lookup_key):
    model_feature_lookups = [FeatureLookup(table_name=table_name, lookup_key=lookup_key)]
    training_set=fe.create_training_set(df=data,feature_lookups=model_feature_lookups,exclude_columns=["src_ip","dst_ip","last_packet","first_packet","packet_len"],label=None)
    training_df = training_set.load_df()
    return training_df

In [0]:
df_train=load_data(df,"dev.feature_store.sessions_arkime","id")
df_train.display()

In [0]:
from pyspark.ml.feature import VectorAssembler, StandardScaler, StringIndexer,CountVectorizer
from pyspark.ml import Pipeline
from pyspark.ml.clustering import KMeans
from pyspark.sql import functions as F
from pyspark.sql.types import FloatType
import numpy as np

In [0]:
num_features = [
    "dst_data_bytes",   
    "tot_data_bytes", 
    "packetlen_max", 
    "packetlen_media", 
    "session_duration"
]

vectorizer = CountVectorizer(inputCol="protocol", outputCol="protocols_vec", binary=True)

assembler = VectorAssembler(
    inputCols=["protocols_vec"] + num_features, 
    outputCol="features_unscaled",
    handleInvalid="skip" 
)

scaler = StandardScaler(
    inputCol="features_unscaled", 
    outputCol="features",
    withStd=True, 
    withMean=True
)

kmeans = KMeans(k=3, seed=42, featuresCol="features", predictionCol="cluster")

pipeline = Pipeline(stages=[vectorizer, assembler, scaler, kmeans])

pipeline_model = pipeline.fit(df_train)
predictions = pipeline_model.transform(df_train)
predictions.display()

In [0]:
kmeans_model = pipeline_model.stages[-1]
centers = kmeans_model.clusterCenters()

def get_distance(features, cluster_idx):
    center = centers[cluster_idx]
    return float(np.linalg.norm(features - center))

distance_udf = F.udf(get_distance, FloatType())

df_dist = predictions.withColumn(
    "dist_to_centroid", 
    distance_udf(F.col("features"), F.col("cluster"))
)

threshold = df_dist.approxQuantile("dist_to_centroid", [0.98], 0.01)[0]

print(f" anomalía detectado: {threshold}")

df_final = df_dist.withColumn("is_anomaly", F.col("dist_to_centroid") > threshold)

df_final.select(
    "protocol", 
    "dst_data_bytes",
    "tot_data_bytes",
    "packetlen_max",
    "packetlen_media",
    "session_duration", 
    "cluster", 
    "dist_to_centroid", 
    "is_anomaly"
).show(100,truncate=False)

In [0]:
import matplotlib.pyplot as plt
cost = []

k_range = range(2, 8)

for k in k_range:
    kmeans = KMeans(k=k, seed=42, featuresCol="features")
    model = kmeans.fit(predictions)
    
    # Obtenemos la inercia
    wcss = model.summary.trainingCost
    cost.append(wcss)
    print(f"Para K={k}, el costo WCSS es: {wcss}")


plt.figure(figsize=(10, 6))
plt.plot(k_range, cost, 'bx-')
plt.xlabel('Número de Clusters')
plt.ylabel('Costo')
plt.title('Método del Codo')
plt.show()

In [0]:
from pyspark.ml.evaluation import ClusteringEvaluator

evaluator = ClusteringEvaluator(
    predictionCol="cluster", 
    featuresCol="features", 
    metricName="silhouette", 
    distanceMeasure="squaredEuclidean"
)
silhouette = evaluator.evaluate(predictions)
print(f"Silhouette : {silhouette}")